**Patch note (Notebook 07):** Exclude zero-norm SVD item vectors before indexing and re-normalize remaining vectors; keep item IDs aligned. This fixes the item-vector normalization assertion error.

# Notebook 07 — FAISS Exact vs ANN Retrieval (MVP)

**Goal:** reuse Notebook 06 split-specific 24-D SVD embeddings; compare `IndexFlatIP` (exact vector-search reference) and `IndexIVFFlat` (approximate), with the same **30-day seen-item exclusions** and **Notebook 02 eligible positives**. Export per-user Top-100 Parquet and reproducible JSON/CSV metrics.

**Important distinction:** `ANN Recall@K` measures overlap with **FAISS FlatIP's filtered exact Top-K**. `Recommendation Recall@K` measures recovery of eligible future positive user-item pairs. Neither measures CTR. FlatIP is exact for the normalized vectors and capped 30k-item embedding catalog, not for the full RetailRocket catalog.

**Windows Conda install** in the *active Jupyter kernel* (run once, then restart kernel):

```python
%pip install faiss-cpu pyarrow duckdb pandas numpy matplotlib
```

If pip wheels are unavailable for your Python/Windows combination, try `conda install -c conda-forge faiss-cpu` in an activated Conda environment. Check `sys.executable` in the next cell. Close earlier notebooks' DuckDB connections. For a predictable laptop workload, FAISS uses four threads by default. No GPU required.

## 1. Setup and validate inputs

In [1]:
from pathlib import Path
import sys, os, time, json
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display
try:
    import faiss
except ImportError as e:
    raise ImportError('FAISS missing in THIS notebook kernel. Run %pip install faiss-cpu, restart kernel, then Run All.') from e

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,*cwd.parents) if (p/'data'/'processed'/'notebook02.duckdb').exists()),None)
if ROOT is None:
    raise FileNotFoundError('Missing data/processed/notebook02.duckdb; open the project root or notebooks/ folder.')
OUT=ROOT/'data'/'processed'; REPORT=ROOT/'data'/'reports'; REPORT.mkdir(parents=True,exist_ok=True)
DB=OUT/'notebook02.duckdb'
con=duckdb.connect(str(DB))
con.execute("SET memory_limit='4GB'");con.execute('SET threads=4')
faiss.omp_set_num_threads(4)
splits=('train','validation','test')
existing={r[0] for r in con.execute('SHOW TABLES').fetchall()}
required={f'{s}_positives' for s in splits}|{f'{s}_history' for s in splits}|{f'{s}_pop_scores' for s in splits}
assert required.issubset(existing), f'Missing DuckDB tables: {required-existing}'
for s in splits:
    for suffix in ('svd_item_embeddings','svd_user_embeddings'):
        p=OUT/f'{s}_{suffix}.parquet'
        assert p.exists(),f'Missing {p}. Run Notebook 06 first.'
print('Kernel:',sys.executable,'FAISS:',getattr(faiss,'__version__','installed'),'DuckDB:',duckdb.__version__)
print('Root:',ROOT)


Kernel: c:\Users\wyr95\.conda\envs\torch\python.exe FAISS: 1.14.3 DuckDB: 1.5.5
Root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks


## 2. Config, exact/ANN definitions

- `IndexFlatIP`: exhaustive inner-product search over normalized item vectors (cosine similarity). Used as the reference **after** excluding seen items.
- `IndexIVFFlat`: coarse k-means partitioning, probing only `nprobe` clusters. Training uses **only the item vectors from the same historical split**; no future events are involved.
- `SEARCH_K`: overfetch before seen filtering. For fairness, **both** indexes request the same `SEARCH_K`. Since a user may have many seen items, overfetch is necessary. The notebook checks whether FlatIP supplies 100 eligible results and flags any shortfall. `SEARCH_K=512` is enough when there are <=30 seen items but we measure actual maximum; adapt dynamically.
- `ANN Recall@K` is micro overlap of user-item pairs with the **filtered FlatIP Top-K**, including users with no embedding (both empty). Also report per-user mean recall over users with nonempty exact reference.
- Timings are batch wall-clock index search + seen filtering + ranking (excluding index build, I/O, and evaluation). Not online p95.

In [2]:
TOP_K=100
KS=(20,50,100)
N_LIST_TARGET=128
NPROBES=(4,16,64)
SEARCH_K_BASE=512
BATCH_USERS=128
SEED=42
EPS=1e-8
RUNS=[];METRICS=[];ANN_METRICS=[];INDEX_STATS=[];SANITY=[]


def qpath(p):
    return "'"+str(p).replace("'","''")+"'"

def read_embeddings(split):
    items=pd.read_parquet(OUT/f'{split}_svd_item_embeddings.parquet')
    users=pd.read_parquet(OUT/f'{split}_svd_user_embeddings.parquet')
    dims=[c for c in items if c.startswith('dim_')]
    assert dims and dims==[c for c in users if c.startswith('dim_')]
    assert items.item_id.is_unique and users.user_id.is_unique
    item_ids=items.item_id.to_numpy(dtype=np.int64)
    user_ids=users.user_id.to_numpy(dtype=np.int64)
    item_vec=np.ascontiguousarray(items[dims].to_numpy(dtype=np.float32))
    user_vec=np.ascontiguousarray(users[dims].to_numpy(dtype=np.float32))
    assert np.isfinite(item_vec).all() and np.isfinite(user_vec).all()
    # Notebook 06 normalizes rows but can legitimately export zero / near-zero SVD rows.
    # Remove zero-norm items BEFORE FAISS indexing; preserve ID-row alignment.
    norms=np.linalg.norm(item_vec,axis=1)
    valid=np.isfinite(norms) & (norms>EPS)
    removed=int((~valid).sum())
    item_ids=item_ids[valid]
    item_vec=np.ascontiguousarray(item_vec[valid]/norms[valid,None],dtype=np.float32)
    assert len(item_ids)>100, 'Too few valid item vectors for Top-100 retrieval'
    # Zero user vectors remain in exported data and are skipped by query_index.
    user_norms=np.linalg.norm(user_vec,axis=1)
    good_users=user_norms>EPS
    user_vec[good_users] /= user_norms[good_users,None]
    user_vec=np.ascontiguousarray(user_vec,dtype=np.float32)
    print(f'{split}: removed {removed:,} zero-norm item vectors; indexed {len(item_ids):,} valid items; '
          f'zero-norm user vectors {int((~good_users).sum()):,}')
    return item_ids,item_vec,user_ids,user_vec


def eval_users_and_seen(split):
    eval_users=con.execute(f'SELECT DISTINCT user_id FROM {split}_positives ORDER BY user_id').df().user_id.to_numpy(dtype=np.int64)
    seen=con.execute(f'''SELECT h.user_id,h.item_id FROM {split}_history h
      JOIN (SELECT DISTINCT user_id FROM {split}_positives) p USING(user_id)''').df()
    seen_map={int(u):set(map(int,g.item_id)) for u,g in seen.groupby('user_id',sort=False)}
    max_seen=max((len(x) for x in seen_map.values()),default=0)
    return eval_users,seen_map,max_seen


## 3. Build FlatIP and IVF indexes

In [3]:
def build_indexes(split,item_vec):
    n,d=item_vec.shape
    t0=time.perf_counter()
    flat=faiss.IndexFlatIP(d)
    flat.add(item_vec)
    flat_sec=time.perf_counter()-t0
    # Conservative number of clusters; IVF training requires enough vectors per centroid.
    nlist=min(N_LIST_TARGET,max(1,n//100))
    t0=time.perf_counter()
    quantizer=faiss.IndexFlatIP(d)
    ivf=faiss.IndexIVFFlat(quantizer,d,int(nlist),faiss.METRIC_INNER_PRODUCT)
    ivf.cp.seed=SEED
    ivf.train(item_vec)
    train_sec=time.perf_counter()-t0
    t0=time.perf_counter();ivf.add(item_vec);add_sec=time.perf_counter()-t0
    assert flat.ntotal==ivf.ntotal==n and ivf.is_trained
    stats=dict(split=split,items=int(n),dimensions=int(d),nlist=int(nlist),
       flat_build_seconds=flat_sec,ivf_train_seconds=train_sec,ivf_add_seconds=add_sec,
       ivf_total_build_seconds=train_sec+add_sec,
       flat_index_bytes=int(len(faiss.serialize_index(flat))),
       ivf_index_bytes=int(len(faiss.serialize_index(ivf))))
    print('Indexes:',stats)
    return flat,ivf,stats


## 4. Query, filter seen items, export candidate Parquet

Query *only* evaluation users with nonzero embeddings. For every method, retrieve `SEARCH_K` vector neighbors, remove the **full Notebook 05/06 30-day seen history**, then take Top-100. The candidate rank is ordered by FAISS score (descending), with item ID as a deterministic tie-breaker. IVF can return fewer than K results if too few lists are probed; this is **reported**, not silently filled with popular items.

The same search budget is used for FlatIP and IVF. The FlatIP reference should return 100 for each active user if `SEARCH_K > max_seen + TOP_K` and catalog size is large enough.

In [4]:
def query_index(split,method,index,item_ids,user_ids,user_vec,eval_users,seen_map,search_k):
    t0=time.perf_counter()
    ix=pd.Index(user_ids).get_indexer(eval_users)
    rows=[]; active_users=0
    for start in range(0,len(eval_users),BATCH_USERS):
        uu=eval_users[start:start+BATCH_USERS]
        loc=ix[start:start+BATCH_USERS]
        valid=loc>=0
        if not valid.any():continue
        ids=uu[valid]
        vec=np.ascontiguousarray(user_vec[loc[valid]],dtype=np.float32)
        active=np.linalg.norm(vec,axis=1)>EPS
        if not active.any():continue
        ids=ids[active];vec=np.ascontiguousarray(vec[active]);active_users+=len(ids)
        scores,positions=index.search(vec,int(search_k))
        for j,u in enumerate(ids):
            excluded=seen_map.get(int(u),set())
            choices=[]
            for pos,score in zip(positions[j],scores[j]):
                if pos<0 or not np.isfinite(score):continue
                item=int(item_ids[int(pos)])
                if item in excluded:continue
                choices.append((item,float(score)))
                if len(choices)>=TOP_K:break
            # Deterministic order for ties in the retrieved subset.
            choices.sort(key=lambda z:(-z[1],z[0]))
            rows.extend((int(u),item,score,rank,method)
                        for rank,(item,score) in enumerate(choices,1))
    seconds=time.perf_counter()-t0
    frame=pd.DataFrame(rows,columns=['user_id','item_id','retrieval_score','retrieval_rank','retrieval_source'])
    table=f'{split}_{method}_candidates'
    con.register('ann_candidates_frame',frame)
    con.execute(f'CREATE OR REPLACE TABLE {table} AS SELECT * FROM ann_candidates_frame')
    con.unregister('ann_candidates_frame')
    path=OUT/f'{split}_{method}_top100.parquet'
    con.execute(f'COPY {table} TO {qpath(path)} (FORMAT PARQUET)')
    covered=int(frame.user_id.nunique())
    short=int((frame.groupby('user_id').size()<TOP_K).sum()) if len(frame) else 0
    info=dict(split=split,method=method,table=table,evaluation_users=len(eval_users),
      nonzero_embedding_users=int(active_users),covered_users=covered,
      user_coverage=covered/len(eval_users) if len(eval_users) else None,
      candidate_rows=len(frame),users_with_fewer_than_100_candidates=short,
      search_k=int(search_k),query_and_filter_seconds=seconds,
      query_ms_per_evaluation_user_amortized=1000*seconds/len(eval_users) if len(eval_users) else None,
      candidates_path=path.name)
    print('Retrieval:',{k:v for k,v in info.items() if k not in ('table','candidates_path')})
    return info


## 5. Evaluation: recommendation Recall, HitRate, Coverage, ANN overlap

In [5]:
def recommendation_metrics(split,table,method):
    positives,users=con.execute(f'SELECT COUNT(*),COUNT(DISTINCT user_id) FROM {split}_positives').fetchone()
    catalog=con.execute(f'SELECT COUNT(*) FROM {split}_pop_scores').fetchone()[0]
    out=[]
    for k in KS:
        hits,hit_users=con.execute(f'''SELECT COUNT(*),COUNT(DISTINCT c.user_id)
          FROM {table} c JOIN {split}_positives p
          ON c.user_id=p.user_id AND c.item_id=p.item_id
          WHERE c.retrieval_rank<={k}''').fetchone()
        macro=con.execute(f'''WITH pos AS (SELECT user_id,COUNT(*) n FROM {split}_positives GROUP BY 1),
          hits AS (SELECT c.user_id,COUNT(*) n FROM {table} c
             JOIN {split}_positives p ON c.user_id=p.user_id AND c.item_id=p.item_id
             WHERE c.retrieval_rank<={k} GROUP BY 1)
          SELECT AVG(COALESCE(h.n,0)::DOUBLE/p.n) FROM pos p LEFT JOIN hits h USING(user_id)''').fetchone()[0]
        unique=con.execute(f'SELECT COUNT(DISTINCT item_id) FROM {table} WHERE retrieval_rank<={k}').fetchone()[0]
        out.append(dict(split=split,method=method,k=k,eligible_positive_pairs=int(positives),
           users=int(users),hits=int(hits),recall_micro=hits/positives if positives else None,
           recall_macro_user=float(macro) if macro is not None else None,
           hit_rate=hit_users/users if users else None,unique_recommended_items=int(unique),
           catalog_coverage=unique/catalog if catalog else None))
    return out


def ann_overlap_metrics(split,method,exact_method='faiss_flat'):
    # Denominator is the number of filtered exact reference user-item pairs at each K.
    # Also calculate mean per-user overlap over users with nonempty exact references.
    out=[]
    for k in KS:
        ref=f'{split}_{exact_method}_candidates'; approx=f'{split}_{method}_candidates'
        reference_pairs=con.execute(f'SELECT COUNT(*) FROM {ref} WHERE retrieval_rank<={k}').fetchone()[0]
        matched=con.execute(f'''SELECT COUNT(*) FROM {ref} a JOIN {approx} b
          ON a.user_id=b.user_id AND a.item_id=b.item_id
          WHERE a.retrieval_rank<={k} AND b.retrieval_rank<={k}''').fetchone()[0]
        mean_user=con.execute(f'''WITH refs AS
          (SELECT user_id,COUNT(*) n FROM {ref} WHERE retrieval_rank<={k} GROUP BY 1),
          matches AS (SELECT a.user_id,COUNT(*) n FROM {ref} a JOIN {approx} b
            ON a.user_id=b.user_id AND a.item_id=b.item_id
            WHERE a.retrieval_rank<={k} AND b.retrieval_rank<={k} GROUP BY 1)
          SELECT AVG(COALESCE(m.n,0)::DOUBLE/r.n) FROM refs r LEFT JOIN matches m USING(user_id)''').fetchone()[0]
        out.append(dict(split=split,method=method,reference=exact_method,k=k,
           reference_pairs=int(reference_pairs),matched_pairs=int(matched),
           ann_recall_micro=matched/reference_pairs if reference_pairs else None,
           ann_recall_macro_user=float(mean_user) if mean_user is not None else None))
    return out


def sanity_checks(split,table,method,expected_users):
    duplicates=con.execute(f'SELECT COUNT(*)-COUNT(DISTINCT (user_id,item_id)) FROM {table}').fetchone()[0]
    seen=con.execute(f'''SELECT COUNT(*) FROM {table} c JOIN {split}_history h
      ON c.user_id=h.user_id AND c.item_id=h.item_id''').fetchone()[0]
    bad_ranks=con.execute(f'''SELECT COUNT(*) FROM
      (SELECT user_id,MIN(retrieval_rank) lo,MAX(retrieval_rank) hi,COUNT(*) n
       FROM {table} GROUP BY 1) WHERE lo<>1 OR hi<>n OR n>{TOP_K}''').fetchone()[0]
    invalid_users=con.execute(f'''SELECT COUNT(*) FROM {table} c
      LEFT JOIN (SELECT DISTINCT user_id FROM {split}_positives) p USING(user_id)
      WHERE p.user_id IS NULL''').fetchone()[0]
    assert duplicates==seen==bad_ranks==invalid_users==0, (duplicates,seen,bad_ranks,invalid_users)
    return dict(split=split,method=method,duplicate_pairs=0,seen_item_leakage=0,bad_ranks=0,
       non_evaluation_users=0)


## 6. Execute Train → Validation → Test

In [6]:
for split in splits:
    print('\n'+'='*24,split.upper(),'='*24)
    item_ids,item_vec,user_ids,user_vec=read_embeddings(split)
    eval_users,seen_map,max_seen=eval_users_and_seen(split)
    # Need > max seen + K to guarantee FlatIP can fill K after exclusions.
    search_k=min(len(item_ids),max(SEARCH_K_BASE,TOP_K+max_seen+1))
    print('Embedding items:',len(item_ids),'eval users:',len(eval_users),
          'max seen:',max_seen,'search_k:',search_k)
    flat,ivf,stat=build_indexes(split,item_vec)
    stat.update(max_seen_items=int(max_seen),search_k=int(search_k))
    INDEX_STATS.append(stat)
    method='faiss_flat'
    r=query_index(split,method,flat,item_ids,user_ids,user_vec,eval_users,seen_map,search_k)
    RUNS.append(r);METRICS.extend(recommendation_metrics(split,r['table'],method))
    SANITY.append(sanity_checks(split,r['table'],method,len(eval_users)))
    assert r['users_with_fewer_than_100_candidates']==0, 'FlatIP should fill Top-100 for all active users. Check embedding catalog and overfetch.'
    for nprobe in NPROBES:
        if nprobe>ivf.nlist:continue
        ivf.nprobe=int(nprobe)
        method=f'faiss_ivf_np{nprobe}'
        r=query_index(split,method,ivf,item_ids,user_ids,user_vec,eval_users,seen_map,search_k)
        RUNS.append(r);METRICS.extend(recommendation_metrics(split,r['table'],method))
        ANN_METRICS.extend(ann_overlap_metrics(split,method))
        SANITY.append(sanity_checks(split,r['table'],method,len(eval_users)))
    del item_ids,item_vec,user_ids,user_vec,flat,ivf

metrics_df=pd.DataFrame(METRICS)
ann_df=pd.DataFrame(ANN_METRICS)
run_df=pd.DataFrame(RUNS)
display(metrics_df[(metrics_df.split=='validation')&(metrics_df.k==100)].round(5))
display(ann_df[(ann_df.split=='validation')&(ann_df.k==100)].round(5))
display(run_df[run_df.split=='validation'][['method','query_and_filter_seconds','users_with_fewer_than_100_candidates']].round(4))



======================== TRAIN ========================
train: removed 5 zero-norm item vectors; indexed 29,995 valid items; zero-norm user vectors 20
Embedding items: 29995 eval users: 4110 max seen: 1586 search_k: 1687
Indexes: {'split': 'train', 'items': 29995, 'dimensions': 24, 'nlist': 128, 'flat_build_seconds': 0.007465700000466313, 'ivf_train_seconds': 0.10200889999941865, 'ivf_add_seconds': 0.006639700000050652, 'ivf_total_build_seconds': 0.1086485999994693, 'flat_index_bytes': 2879565, 'ivf_index_bytes': 3132931}
Retrieval: {'split': 'train', 'method': 'faiss_flat', 'evaluation_users': 4110, 'nonzero_embedding_users': 3285, 'covered_users': 3285, 'user_coverage': 0.7992700729927007, 'candidate_rows': 328500, 'users_with_fewer_than_100_candidates': 0, 'search_k': 1687, 'query_and_filter_seconds': 1.007565699999759, 'query_ms_per_evaluation_user_amortized': 0.24514980535273945}
Retrieval: {'split': 'train', 'method': 'faiss_ivf_np4', 'evaluation_users': 4110, 'nonzero_embedding

,split,method,k,eligible_positive_pairs,users,hits,recall_micro,recall_macro_user,hit_rate,unique_recommended_items,catalog_coverage
14,validation,faiss_flat,100,9020,3657,355,0.03936,0.04618,0.07274,29162,0.24779
17,validation,faiss_ivf_np4,100,9020,3657,357,0.03958,0.04569,0.07274,28707,0.24393
20,validation,faiss_ivf_np16,100,9020,3657,351,0.03891,0.04573,0.07164,29067,0.24699
23,validation,faiss_ivf_np64,100,9020,3657,355,0.03936,0.04618,0.07274,29155,0.24773


,split,method,reference,k,reference_pairs,matched_pairs,ann_recall_micro,ann_recall_macro_user
11,validation,faiss_ivf_np4,faiss_flat,100,289000,260798,0.90242,0.90242
14,validation,faiss_ivf_np16,faiss_flat,100,289000,286403,0.99101,0.99101
17,validation,faiss_ivf_np64,faiss_flat,100,289000,288910,0.99969,0.99969


,method,query_and_filter_seconds,users_with_fewer_than_100_candidates
4,faiss_flat,0.8710,0
5,faiss_ivf_np4,0.5582,0
6,faiss_ivf_np16,0.6361,0
7,faiss_ivf_np64,0.6843,0


## 7. Compare against Notebook 06 SVD and prior retrieval baselines

The FAISS FlatIP reference should match Notebook 06 SVD Top-100 *closely* because it uses the same normalized embeddings. However, tie ordering and floating-point implementation may differ, especially among near-equal scores. This notebook measures exact filtered vector search separately rather than assuming identical output.

In [7]:
reports=[]
for name in ('notebook04_popularity_report.json','notebook05_item_cf_report.json','notebook06_svd_report.json'):
    p=REPORT/name
    if p.exists():
        reports.append(pd.DataFrame(json.loads(p.read_text(encoding='utf-8'))['metrics']))
if reports:
    comparison=pd.concat([metrics_df,*reports],ignore_index=True)
    display(comparison[(comparison.split=='validation')&(comparison.k==100)][
        ['method','recall_micro','hit_rate','catalog_coverage']].sort_values('recall_micro',ascending=False).round(5))
else:
    print('Previous reports optional; no baseline JSON files found.')


,method,recall_micro,hit_rate,catalog_coverage
65,cooccurrence,0.08736,0.13946,0.22630
71,jaccard,0.08237,0.13344,0.25852
68,cosine,0.07672,0.12606,0.26158
17,faiss_ivf_np4,0.03958,0.07274,0.24393
14,faiss_flat,0.03936,0.07274,0.24779
23,faiss_ivf_np64,0.03936,0.07274,0.24773
86,svd,0.03936,0.07274,0.24779
20,faiss_ivf_np16,0.03891,0.07164,0.24699
47,decayed,0.03670,0.06262,0.00140
44,global,0.03415,0.05934,0.00146


## 8. Save CSV, JSON and candidate Parquet files

In [8]:
metrics_df.to_csv(REPORT/'notebook07_recommendation_metrics.csv',index=False)
ann_df.to_csv(REPORT/'notebook07_ann_recall_metrics.csv',index=False)
run_df.to_csv(REPORT/'notebook07_latency_metrics.csv',index=False)

def safe(x):
    if isinstance(x,dict):return {str(k):safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)):return [safe(v) for v in x]
    if isinstance(x,np.integer):return int(x)
    if isinstance(x,np.floating):return float(x) if np.isfinite(x) else None
    if isinstance(x,np.bool_):return bool(x)
    return x

report=dict(notebook='07_faiss_ann_retrieval',algorithm='L2-normalized SVD embeddings + FAISS IndexFlatIP / IndexIVFFlat',
  max_k=TOP_K,ks=list(KS),nlist_target=N_LIST_TARGET,nprobes=list(NPROBES),
  batch_users=BATCH_USERS,search_k_base=SEARCH_K_BASE,seed=SEED,
  evaluation='Same capped eligible positives and 30-day seen filtering as Notebooks 04-06',
  ann_reference='Filtered FAISS FlatIP Top-K on identical 30k-capped normalized item vectors',
  index_stats=INDEX_STATS,runs=RUNS,recommendation_metrics=METRICS,ann_metrics=ANN_METRICS,sanity=SANITY,
  limitations=[
    'Exact reference is FlatIP over Notebook 06 capped embedding item universe, not the entire item catalog',
    'IVF candidate shortage after filtering is reported; no popularity fallback or exact repair',
    'Per-split SVD embeddings were trained in Notebook 06 using pre-cutoff interactions; no new fitting to future positives',
    'ANN Recall is overlap with exact vector search; recommendation Recall is recovery of capped eligible future positives',
    'Batch query timings include Python filtering/ranking but exclude index construction, Parquet I/O and production API overhead',
    'No impression logs; recommendation metrics are not actual CTR/CVR or causal impact',
    'Model selection should use validation; test is for confirmation'])
p=REPORT/'notebook07_faiss_report.json'
p.write_text(json.dumps(safe(report),indent=2,allow_nan=False),encoding='utf-8')
print('Report:',p)
print('CSV metrics:',REPORT/'notebook07_recommendation_metrics.csv',REPORT/'notebook07_ann_recall_metrics.csv')
print('Candidates:',OUT/'{split}_faiss_flat_top100.parquet','and IVF nprobe variants')
con.close()


Report: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook07_faiss_report.json
CSV metrics: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook07_recommendation_metrics.csv C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook07_ann_recall_metrics.csv
Candidates: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\processed\{split}_faiss_flat_top100.parquet and IVF nprobe variants


## 9. Handoff — Notebook 08 Multi-Source Retrieval Fusion

Upload `data/reports/notebook07_faiss_report.json`. Next, merge **Popularity + Item-CF + Embedding/FAISS** candidates, deduplicate `(user_id,item_id)`, and measure **union Recall@K** plus source complementarity. Avoid claiming IVF improves ranking quality just because it changes the candidate order; ANN is primarily an approximate-search efficiency trade-off.